# GLM-5.2 Tensor Quantization Analysis

Read `tensors.txt` and group every checkpoint tensor by its on-disk dtype.

In [32]:
from collections import defaultdict
from pathlib import Path

tensor_file = Path("tensors.txt")
if not tensor_file.exists():
    raise FileNotFoundError(
        f"{tensor_file.resolve()} does not exist. Start Jupyter in the notebook directory."
    )

tensors_by_dtype = defaultdict(list)
for line_number, line in enumerate(tensor_file.read_text().splitlines(), start=1):
    if not line.strip():
        continue
    try:
        name, dtype, shape = line.split(maxsplit=2)
    except ValueError as exc:
        raise ValueError(f"Malformed line {line_number}: {line!r}") from exc
    tensors_by_dtype[dtype].append((name, shape))

expected_dtypes = {"F8_E4M3", "F32", "BF16"}
unexpected_dtypes = set(tensors_by_dtype) - expected_dtypes
if unexpected_dtypes:
    raise ValueError(f"Unexpected dtypes: {sorted(unexpected_dtypes)}")

print(f"Loaded {sum(map(len, tensors_by_dtype.values())):,} tensors from {tensor_file}")
for dtype in ("F8_E4M3", "F32", "BF16"):
    print(f"{dtype}: {len(tensors_by_dtype[dtype]):,}")

Loaded 118,629 tensors from tensors.txt
F8_E4M3: 59,044
F32: 59,120
BF16: 465


In [33]:
print(f"FP8 tensors ({len(tensors_by_dtype['F8_E4M3']):,})")
for name, shape in tensors_by_dtype["F8_E4M3"][:50]:
    print(f"{name} {shape}")

FP8 tensors (59,044)
model.layers.0.mlp.down_proj.weight [6144, 12288]
model.layers.0.mlp.gate_proj.weight [12288, 6144]
model.layers.0.mlp.up_proj.weight [12288, 6144]
model.layers.0.self_attn.indexer.wk.weight [128, 6144]
model.layers.0.self_attn.indexer.wq_b.weight [4096, 2048]
model.layers.0.self_attn.kv_a_proj_with_mqa.weight [576, 6144]
model.layers.0.self_attn.kv_b_proj.weight [28672, 512]
model.layers.0.self_attn.o_proj.weight [6144, 16384]
model.layers.0.self_attn.q_a_proj.weight [2048, 6144]
model.layers.0.self_attn.q_b_proj.weight [16384, 2048]
model.layers.1.mlp.down_proj.weight [6144, 12288]
model.layers.1.mlp.gate_proj.weight [12288, 6144]
model.layers.1.mlp.up_proj.weight [12288, 6144]
model.layers.1.self_attn.indexer.wk.weight [128, 6144]
model.layers.1.self_attn.indexer.wq_b.weight [4096, 2048]
model.layers.1.self_attn.kv_a_proj_with_mqa.weight [576, 6144]
model.layers.1.self_attn.kv_b_proj.weight [28672, 512]
model.layers.1.self_attn.o_proj.weight [6144, 16384]
model.

In [34]:
import re

def fp8_pattern(name: str) -> str:
    # Drop "model.layers.<N>." and collapse "experts.<k>." to "experts.*."
    # so per-layer / per-expert duplicates fold into one entry.
    name = re.sub(r"^model\.layers\.\d+\.", "", name)
    return re.sub(r"\.experts\.\d+\.", ".experts.*.", name)

patterns = {}
for name, shape in tensors_by_dtype["F8_E4M3"]:
    key = (fp8_pattern(name), shape)
    patterns[key] = patterns.get(key, 0) + 1

print(f"Unique FP8 tensor patterns ({len(patterns)} of {len(tensors_by_dtype['F8_E4M3']):,} tensors)")
for (name, shape), count in sorted(patterns.items()):
    print(f"{name} {shape}  x{count}")

Unique FP8 tensor patterns (16 of 59,044 tensors)
mlp.down_proj.weight [6144, 12288]  x3
mlp.experts.*.down_proj.weight [6144, 2048]  x19456
mlp.experts.*.gate_proj.weight [2048, 6144]  x19456
mlp.experts.*.up_proj.weight [2048, 6144]  x19456
mlp.gate_proj.weight [12288, 6144]  x3
mlp.shared_experts.down_proj.weight [6144, 2048]  x76
mlp.shared_experts.gate_proj.weight [2048, 6144]  x76
mlp.shared_experts.up_proj.weight [2048, 6144]  x76
mlp.up_proj.weight [12288, 6144]  x3
self_attn.indexer.wk.weight [128, 6144]  x22
self_attn.indexer.wq_b.weight [4096, 2048]  x22
self_attn.kv_a_proj_with_mqa.weight [576, 6144]  x79
self_attn.kv_b_proj.weight [28672, 512]  x79
self_attn.o_proj.weight [6144, 16384]  x79
self_attn.q_a_proj.weight [2048, 6144]  x79
self_attn.q_b_proj.weight [16384, 2048]  x79


In [35]:
print(f"FP32 tensors ({len(tensors_by_dtype['F32']):,})")
for name, shape in tensors_by_dtype["F32"][:50]:
    print(f"{name} {shape}")

FP32 tensors (59,120)
model.layers.0.mlp.down_proj.weight_scale_inv [48, 96]
model.layers.0.mlp.gate_proj.weight_scale_inv [96, 48]
model.layers.0.mlp.up_proj.weight_scale_inv [96, 48]
model.layers.0.self_attn.indexer.wk.weight_scale_inv [1, 48]
model.layers.0.self_attn.indexer.wq_b.weight_scale_inv [32, 16]
model.layers.0.self_attn.kv_a_proj_with_mqa.weight_scale_inv [5, 48]
model.layers.0.self_attn.kv_b_proj.weight_scale_inv [224, 4]
model.layers.0.self_attn.o_proj.weight_scale_inv [48, 128]
model.layers.0.self_attn.q_a_proj.weight_scale_inv [16, 48]
model.layers.0.self_attn.q_b_proj.weight_scale_inv [128, 16]
model.layers.1.mlp.down_proj.weight_scale_inv [48, 96]
model.layers.1.mlp.gate_proj.weight_scale_inv [96, 48]
model.layers.1.mlp.up_proj.weight_scale_inv [96, 48]
model.layers.1.self_attn.indexer.wk.weight_scale_inv [1, 48]
model.layers.1.self_attn.indexer.wq_b.weight_scale_inv [32, 16]
model.layers.1.self_attn.kv_a_proj_with_mqa.weight_scale_inv [5, 48]
model.layers.1.self_att

In [36]:
print(f"BF16 tensors ({len(tensors_by_dtype['BF16']):,})")
for name, shape in tensors_by_dtype["BF16"][:50]:
    print(f"{name} {shape}")

BF16 tensors (465)
lm_head.weight [154880, 6144]
model.embed_tokens.weight [154880, 6144]
model.layers.0.input_layernorm.weight [6144]
model.layers.0.post_attention_layernorm.weight [6144]
model.layers.0.self_attn.indexer.k_norm.bias [128]
model.layers.0.self_attn.indexer.k_norm.weight [128]
model.layers.0.self_attn.indexer.weights_proj.weight [32, 6144]
model.layers.0.self_attn.kv_a_layernorm.weight [512]
model.layers.0.self_attn.q_a_layernorm.weight [2048]
model.layers.1.input_layernorm.weight [6144]
model.layers.1.post_attention_layernorm.weight [6144]
model.layers.1.self_attn.indexer.k_norm.bias [128]
model.layers.1.self_attn.indexer.k_norm.weight [128]
model.layers.1.self_attn.indexer.weights_proj.weight [32, 6144]
model.layers.1.self_attn.kv_a_layernorm.weight [512]
model.layers.1.self_attn.q_a_layernorm.weight [2048]
model.layers.10.input_layernorm.weight [6144]
model.layers.10.mlp.gate.weight [256, 6144]
model.layers.10.post_attention_layernorm.weight [6144]
model.layers.10.sel

In [37]:
import ast
import math
from collections import Counter, defaultdict

ROUTED_EXPERT_MARKER = ".mlp.experts."
BYTES_PER_DTYPE = {"F8_E4M3": 1, "BF16": 2, "F32": 4}

stats = defaultdict(lambda: {"tensors": 0, "weights": 0, "loaded_bytes": 0})
loaded_dtype_breakdown = defaultdict(Counter)

for checkpoint_dtype, tensors in tensors_by_dtype.items():
    for name, shape_text in tensors:
        shape = ast.literal_eval(shape_text)
        weights = math.prod(shape)
        category = "routed experts" if ROUTED_EXPERT_MARKER in name else "dense"

        # Only routed-expert FP8 tensors remain FP8 after loading.
        loaded_dtype = (
            "F8_E4M3"
            if checkpoint_dtype == "F8_E4M3" and category == "routed experts"
            else "BF16"
            if checkpoint_dtype == "F8_E4M3"
            else checkpoint_dtype
        )

        stats[category]["tensors"] += 1
        stats[category]["weights"] += weights
        stats[category]["loaded_bytes"] += weights * BYTES_PER_DTYPE[loaded_dtype]
        loaded_dtype_breakdown[category][loaded_dtype] += weights

for category in ("routed experts", "dense"):
    values = stats[category]
    print(f"{category}:")
    print(f"  tensors: {values['tensors']:,}")
    print(f"  weights: {values['weights']:,} ({values['weights'] / 1e9:.3f}B)")
    print(f"  loaded size: {values['loaded_bytes'] / 1e9:.3f} GB")
    print("  loaded dtype breakdown:")
    for dtype in ("F8_E4M3", "BF16", "F32"):
        weights = loaded_dtype_breakdown[category][dtype]
        if weights:
            size_gb = weights * BYTES_PER_DTYPE[dtype] / 1e9
            print(f"    {dtype}: {weights:,} weights ({size_gb:.3f} GB)")

routed experts:
  tensors: 116,736
  weights: 734,484,234,240 (734.484B)
  loaded size: 734.619 GB
  loaded dtype breakdown:
    F8_E4M3: 734,439,407,616 weights (734.439 GB)
    F32: 44,826,624 weights (0.179 GB)
dense:
  tensors: 1,893
  weights: 18,891,559,344 (18.892B)
  loaded size: 37.785 GB
  loaded dtype breakdown:
    BF16: 18,890,513,408 weights (37.781 GB)
    F32: 1,045,936 weights (0.004 GB)


In [ ]:
dense_weights = 37.785
routed_expert_weights = 734.619


NUM_LAYERS = 78
HIDDEN_DIM = 6144

checkpointed_activation_memory_per_token = (
    NUM_LAYERS * HIDDEN_DIM * BYTES_PER_DTYPE["BF16"] / 1e9
)

VOCAB = 154_880
LM_HEAD_CHUNK = 4096  # CHUNKED_LM_HEAD_SEQ_CHUNK in chunked_lm_head.py


def current_lm_head_memspike(TP: int, CP: int, seqlen: int):
    """Peak GB held by the LM head on one rank, as implemented today.

    Measured from the 131K memory profile (glm52_b300_cp8ep8pp1_memory_20260831):
    1. fp32 casts of the frozen head weight, one PER CHUNK, each saved by
       autograd for the matmul backward (chunked_lm_head.py:183).
    2. fp32 logits saved by the cross-entropy for backward. Chunking does
       NOT shrink these: they accumulate across all chunks.
    3. One chunk of in-flight transients at the peak: matmul output +
       fp32-promoted LoRA delta + bf16 LoRA delta.

    TP shards the vocab dim of weight and logits; CP shards the tokens.
    Sanity: current_lm_head_memspike(1, 8, 131072) == 31.7 GB == 29.5 GiB,
    matching the profiled peak.
    """
    tokens = seqlen / CP
    vocab = VOCAB / TP
    chunks = math.ceil(tokens / LM_HEAD_CHUNK)

    weight_casts = chunks * vocab * HIDDEN_DIM * BYTES_PER_DTYPE["F32"]
    saved_logits = tokens * vocab * BYTES_PER_DTYPE["F32"]
    transient_chunk = (
        LM_HEAD_CHUNK * vocab * (2 * BYTES_PER_DTYPE["F32"] + BYTES_PER_DTYPE["BF16"])
    )

    return (weight_casts + saved_logits + transient_chunk) / 1e9


def fixed_lm_head_memspike(TP: int, CP: int, seqlen: int):
    """Peak GB per rank with the two LPS-1208 fixes:
    https://linear.app/baseten/issue/LPS-1208/investigate-and-optimize-lm-head-memory-usage-in-trainers-affects-all

    1. Hoist the weight cast out of the chunk loop: a single fp32 copy of
       the head weight, held from head-forward until head-backward.
    2. Fused cross-entropy that recomputes logits per chunk in backward
       (Liger / Cut-Cross-Entropy style): at most one chunk of logits
       exists at a time, plus one chunk of backward workspace.

    ~Flat in seqlen. Hoist-only variant: add back
    seqlen / CP * VOCAB / TP * 4 bytes of saved logits.
    """
    vocab = VOCAB / TP

    weight_cast = vocab * HIDDEN_DIM * BYTES_PER_DTYPE["F32"]
    live_logits = 2 * LM_HEAD_CHUNK * vocab * BYTES_PER_DTYPE["F32"]

    return (weight_cast + live_logits) / 1e9


def moe_layer_backward_memspike(TP: int, EP: int, CP: int, seqlen: int):
    materialized_bf16_weights = 6144 * 2048 * 2 * 3 * (256 / EP) / 1e9

    # includes the s
    dsa_state = (
        (4.569 / 16384) * (seqlen / CP) * (1 / TP)
    )  # note idk if this is accurate for TP.

    # Measured live tensors were:
    # - Dispatched expert input: [C, 6144]
    # - FC1 combined gate/up output: [C, 4096]
    # - Gate/up materializations: two [C, 2048]
    # - SiLU output: [C, 2048]
    # - Elementwise product: [C, 2048]
    # - FC2 output: [C, 6144]

    topk = 8
    routing_imbalance_penalty = 2.5  # as a multiple of 'even'. lets be conservative.
    moe_sublayer_activations = routing_imbalance_penalty * topk * (seqlen / CP) * 2 * (2 * 6144 + 6 * 2048) / 1e9

    return materialized_bf16_weights + dsa_state + moe_sublayer_activations


def memory_usage_per_rank(TP: int, CP: int, EP: int, seqlen: int):
    dense = dense_weights / TP
    experts = routed_expert_weights / EP
    checkpointed_activations = checkpointed_activation_memory_per_token * seqlen / CP

    lmhead_spike = current_lm_head_memspike(TP, CP, seqlen)
    moelayer_spike = moe_layer_backward_memspike(TP, EP, CP, seqlen)
    return dense + experts + checkpointed_activations + max(lmhead_spike,moelayer_spike)


cpep = [(8, 8), (16, 8), (16,16), (32, 32)]
seqlens = [131072 * 1, 131072 * 2, 131072 * 4]

for cp, ep in cpep:
    print(f"CP={cp}, EP={ep}")

    for s in seqlens:
        print(f"seqlen {s}: {memory_usage_per_rank(1, cp, ep, s)}")

moe_layer_backward_memspike(1,8,8, 131072)


CP=8, EP=8
seqlen 131072: 177.03527317599998
seqlen 262144: 218.11428655199998
seqlen 524288: 300.272313304
CP=16, EP=8
seqlen 131072: 156.495766488
seqlen 262144: 177.03527317599998
seqlen 524288: 218.11428655199998
CP=16, EP=16
seqlen 131072: 110.582078988
seqlen 262144: 131.121585676
seqlen 524288: 172.200599052
CP=32, EP=32
seqlen 131072: 77.355481894
seqlen 262144: 87.62523523800002
seqlen 524288: 108.164741926


23.091046463999998

In [39]:
# Actual values from basetenlabs/trainers PR #1243, measured 31AUG.
# The source measurements are GiB. Convert them to decimal GB to match
# memory_usage_per_rank.
actual_values = [
    # tokens, control TPS/GPU, stable TPS/GPU, allocated GiB, reserved GiB
    (131_072, 1179, 1196, 171.06, 174.38),
    (196_608, 1197, 1192, 192.19, 196.82),
    (262_144, 1189, 1187, 213.32, 218.55),
    (327_680, 1081, 1077, 234.45, 240.94),
    (393_216, 913, 879, 255.58, 257.54),
]

ACTUAL_TP = 1
ACTUAL_CP = 8
ACTUAL_EP = 8
GIB_TO_GB = 2**30 / 1e9

headers = (
    "Exact tokens",
    "Control TPS/GPU",
    "Stable TPS/GPU",
    "Peak allocated (GB)",
    "Peak reserved (GB)",
    "Estimated (GB)",
)
rows = []
for tokens, control_tps, stable_tps, allocated_gib, reserved_gib in actual_values:
    rows.append(
        (
            f"{tokens:,}",
            str(control_tps),
            str(stable_tps),
            f"{allocated_gib * GIB_TO_GB:.2f}",
            f"{reserved_gib * GIB_TO_GB:.2f}",
            f"{memory_usage_per_rank(ACTUAL_TP, ACTUAL_CP, ACTUAL_EP, tokens):.2f}",
        )
    )

widths = [max(len(headers[i]), *(len(row[i]) for row in rows)) for i in range(len(headers))]
row_format = " | ".join(f"{{:<{width}}}" for width in widths)

print(f"Actual configuration: TP={ACTUAL_TP}, CP={ACTUAL_CP}, EP={ACTUAL_EP}")
print(row_format.format(*headers))
print("-+-".join("-" * width for width in widths))
for row in rows:
    print(row_format.format(*row))


Actual configuration: TP=1, CP=8, EP=8
Exact tokens | Control TPS/GPU | Stable TPS/GPU | Peak allocated (GB) | Peak reserved (GB) | Estimated (GB)
-------------+-----------------+----------------+---------------------+--------------------+---------------
131,072      | 1179            | 1196           | 183.67              | 187.24             | 177.04        
196,608      | 1197            | 1192           | 206.36              | 211.33             | 197.57        
262,144      | 1189            | 1187           | 229.05              | 234.67             | 218.11        
327,680      | 1081            | 1077           | 251.74              | 258.71             | 238.65        
393,216      | 913             | 879            | 274.43              | 276.53             | 259.19        
